# PRNU-v2 reference-free binary usefulness gate

Run this notebook from **Runtime > Run all** in a Colab GPU runtime. It rebuilds the selected matched-clean and independent robustness views on Colab-local storage, restores the three controlled-RINE parent checkpoints from Drive, extracts the reference-free PRNU-v2 vector, trains the PRNU-only diagnostic and RINE+PRNU candidate for seeds 42/43/44, applies the locked retention gate, and syncs only durable outputs. It never reads `final_test`, never treats known-device PCE as an authenticity score, and never copies the 19,460-image transform cache back to Drive.

Expected Drive inputs:

- `MyDrive/hackathon_data/raw/sid_set/images`
- `MyDrive/cya-techjam26/artifacts/task2/fixed_q96_manifest.csv`
- `MyDrive/cya-techjam26/artifacts/task2/source_manifest_split.csv`
- `MyDrive/cya-techjam26/artifacts/robustness/train-controlled-rine/seed_{42,43,44}`


In [ ]:
import subprocess
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
REPO_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'
if PROJECT.is_dir():
    git_result = subprocess.run(
        ['git', 'pull', '--ff-only'], cwd=PROJECT, text=True
    )
else:
    git_result = subprocess.run(
        ['git', 'clone', REPO_URL, str(PROJECT)], text=True
    )
assert git_result.returncode == 0, 'Repository clone/pull failed.'
required_files = (
    PROJECT / 'scripts/extract_prnu_runtime_v2.py',
    PROJECT / 'scripts/train_prnu_runtime_v2.py',
    PROJECT / 'scripts/run_robustness_fusion.py',
    PROJECT / 'scripts/compare_robustness_candidate.py',
    PROJECT / 'src/cya_detector/features/prnu_runtime_v2.py',
)
missing_files = [str(path.relative_to(PROJECT)) for path in required_files if not path.is_file()]
assert not missing_files, (
    'The checkout does not contain the Notebook 08 implementation. Commit and push '
    f'the PRNU-v2 changes first. Missing: {missing_files}'
)
print('Repository ready:', PROJECT)


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
import subprocess
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')

def run_command(*args):
    result = subprocess.run([str(value) for value in args], cwd=PROJECT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, args))}")
    return result

run_command('make', 'install-colab')
import torch
assert torch.cuda.is_available(), (
    'Notebook 08 requires a GPU runtime. In Colab select Runtime > Change runtime '
    'type > GPU, then use Runtime > Run all.'
)
print('GPU:', torch.cuda.get_device_name(0))
run_command('make', 'smoke-bootstrap')
run_command(
    'python', '-m', 'unittest',
    'tests.test_prnu_runtime_v2',
    'tests.test_robustness_training',
    '-v',
)
print('Environment and targeted tests passed.')


In [ ]:
import csv
import hashlib
import json
import shutil
import subprocess
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
PRIOR = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
DURABLE = PRIOR / 'robustness'
ROBUSTNESS = PROJECT / 'artifacts/robustness'
SEEDS = (42, 43, 44)
fixed_manifest = PRIOR / 'task2/fixed_q96_manifest.csv'
source_manifest = PRIOR / 'task2/source_manifest_split.csv'
drive_images = Path('/content/drive/MyDrive/hackathon_data/raw/sid_set/images')
local_images = Path('/content/hackathon_data/raw/sid_set/images')
regenerated_manifest = PROJECT / 'artifacts/task2/fixed_q96_manifest_regenerated.csv'

def run_command(*args):
    result = subprocess.run([str(value) for value in args], cwd=PROJECT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, args))}")
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

for required in (fixed_manifest, source_manifest, drive_images, DURABLE):
    assert required.exists(), f'Required Drive input not found: {required}'

with fixed_manifest.open(newline='') as stream:
    fixed_rows = list(csv.DictReader(stream))
assert len(fixed_rows) == 2000, f'Expected 2,000 selected rows, found {len(fixed_rows)}'
assert Counter(row['label'] for row in fixed_rows) == {'authentic': 1000, 'ai_generated': 1000}
selected_source_ids = {row['source_id'] for row in fixed_rows}
assert len(selected_source_ids) == len(fixed_rows), 'Selected source IDs are not unique'

def stage_selected_source(row):
    filename = Path(row['source_path']).name
    source = drive_images / filename
    destination = local_images / filename
    assert source.is_file(), f'Selected raw source missing from Drive: {source}'
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        temporary = destination.with_suffix(destination.suffix + '.part')
        shutil.copy2(source, temporary)
        temporary.replace(destination)
    return destination

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(stage_selected_source, row) for row in fixed_rows]
    for completed, future in enumerate(as_completed(futures), start=1):
        future.result()
        if completed % 250 == 0 or completed == len(futures):
            print(f'Staged {completed}/{len(futures)} selected raw sources')

with source_manifest.open(newline='') as stream:
    reader = csv.DictReader(stream)
    source_fields = reader.fieldnames
    selected_source_rows = [row for row in reader if row['source_id'] in selected_source_ids]
assert source_fields and 'source_path' in source_fields
assert len(selected_source_rows) == 2000, (
    f'Expected 2,000 selected source records, found {len(selected_source_rows)}'
)
for row in selected_source_rows:
    row['source_path'] = str(local_images / Path(row['source_path']).name)
local_source_manifest = PROJECT / 'artifacts/task2/source_manifest_split_local.csv'
local_source_manifest.parent.mkdir(parents=True, exist_ok=True)
with local_source_manifest.open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=source_fields)
    writer.writeheader()
    writer.writerows(selected_source_rows)

run_command(
    'python', 'scripts/build_matched_clean.py',
    '--source-manifest', local_source_manifest,
    '--output-root', PROJECT / 'artifacts/task2/matched_candidates',
    '--output-manifest', regenerated_manifest,
    '--report', PROJECT / 'artifacts/task2/fixed_q96_report_regenerated.json',
    '--policy', 'fixed_q96', '--seed', '42', '--limit-per-label', '1000',
)
with regenerated_manifest.open(newline='') as stream:
    regenerated_rows = list(csv.DictReader(stream))
fixed_by_source = {row['source_id']: row for row in fixed_rows}
regenerated_by_source = {row['source_id']: row for row in regenerated_rows}
assert regenerated_by_source.keys() == fixed_by_source.keys(), 'Regenerated source set changed'
byte_mismatches = [
    source_id for source_id in fixed_by_source
    if regenerated_by_source[source_id]['sha256'] != fixed_by_source[source_id]['sha256']
]
assert not byte_mismatches, (
    'Regenerated fixed-Q96 bytes differ from Notebook 07 inputs; stop before comparing '
    f'predictions. First mismatches: {byte_mismatches[:5]}'
)

combined_manifest = ROBUSTNESS / 'manifests/combined_manifest.csv'
clean_report_path = ROBUSTNESS / 'manifests/clean_manifest_report.json'
reuse_bank = False
if combined_manifest.is_file() and clean_report_path.is_file():
    clean_report = json.loads(clean_report_path.read_text())
    if clean_report.get('input_manifest_sha256') == sha256_file(regenerated_manifest):
        with combined_manifest.open(newline='') as stream:
            existing_rows = list(csv.DictReader(stream))
        reuse_bank = bool(existing_rows) and all(Path(row['image_path']).is_file() for row in existing_rows)
if reuse_bank:
    print(f'Reusing complete local robustness bank: {len(existing_rows):,} rows')
else:
    run_command(
        'make', 'robustness-prepare',
        f'TASK2_SELECTED_MANIFEST={regenerated_manifest}',
    )

parent_filenames = ('best_50_50.pt', 'best_50_50_predictions.csv')
for seed in SEEDS:
    durable_seed = DURABLE / 'train-controlled-rine' / f'seed_{seed}'
    local_seed = ROBUSTNESS / 'train-controlled-rine' / f'seed_{seed}'
    local_seed.mkdir(parents=True, exist_ok=True)
    for filename in parent_filenames:
        source = durable_seed / filename
        assert source.is_file(), f'Notebook 07 parent artifact not found: {source}'
        shutil.copy2(source, local_seed / filename)

assert combined_manifest.is_file(), f'Combined robustness manifest missing: {combined_manifest}'
with combined_manifest.open(newline='') as stream:
    combined_rows = list(csv.DictReader(stream))
assert combined_rows, 'Combined robustness manifest is empty'
assert all(row['split'] != 'final_test' for row in combined_rows), 'final_test leaked into Notebook 08'
allowed_roots = (PROJECT / 'artifacts/task2/matched_candidates', ROBUSTNESS / 'variants')
unexpected = [
    row['image_path'] for row in combined_rows
    if not any(Path(row['image_path']).is_relative_to(root) for root in allowed_roots)
]
missing = [row['image_path'] for row in combined_rows if not Path(row['image_path']).is_file()]
assert not unexpected, f'Unexpected manifest paths: {unexpected[:5]}'
assert not missing, f'Missing manifest images: {missing[:5]} (total={len(missing)})'
print(f'PASS: local bank has {len(combined_rows):,} rows and all parent artifacts are restored.')


In [ ]:
import json
import subprocess
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
ROBUSTNESS = PROJECT / 'artifacts/robustness'
SEEDS = (42, 43, 44)

def run_make(target, *assignments):
    command = ['make', target, *assignments]
    result = subprocess.run(command, cwd=PROJECT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}")
    return result

combined_manifest = ROBUSTNESS / 'manifests/combined_manifest.csv'
assert combined_manifest.is_file(), 'Preparation did not produce the combined manifest'
for seed in SEEDS:
    parent = ROBUSTNESS / 'train-controlled-rine' / f'seed_{seed}' / 'best_50_50.pt'
    assert parent.is_file(), f'Controlled-RINE parent missing: {parent}'

run_make('robustness-prnu-v2-extract')
readiness_path = ROBUSTNESS / 'features/prnu_v2_runtime_extraction_report.json'
readiness = json.loads(readiness_path.read_text())
assert readiness['ready_for_binary_ablation'], readiness
assert not readiness['reference_comparison_used']
assert not readiness['final_test_read']
print('PRNU-v2 readiness groups:', json.dumps(readiness['groups'], indent=2))

for seed in SEEDS:
    print(f'PRNU-only diagnostic seed {seed}')
    run_make('robustness-prnu-v2-train', f'ROBUSTNESS_SEED={seed}')
for seed in SEEDS:
    print(f'RINE+PRNU fusion seed {seed}')
    run_make('robustness-prnu-v2-fusion', f'ROBUSTNESS_SEED={seed}')

run_make('robustness-prnu-v2-compare')
decision_path = ROBUSTNESS / 'reports/prnu_v2/retention_decision.json'
decision = json.loads(decision_path.read_text())
assert decision['decision'] in {'retain', 'reject'}
assert not decision['final_test_read']
print(json.dumps(decision, indent=2, sort_keys=True))


In [ ]:
import hashlib
import shutil
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
ROBUSTNESS = PROJECT / 'artifacts/robustness'
DURABLE = Path('/content/drive/MyDrive/cya-techjam26/artifacts/robustness')
decision_path = ROBUSTNESS / 'reports/prnu_v2/retention_decision.json'
assert decision_path.is_file(), 'PRNU-v2 decision is missing; the ablation did not finish'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def copy_file_verified(local_path, remote_path):
    remote_path.parent.mkdir(parents=True, exist_ok=True)
    local_hash = sha256_file(local_path)
    if not remote_path.is_file() or sha256_file(remote_path) != local_hash:
        temporary = remote_path.with_suffix(remote_path.suffix + '.part')
        shutil.copy2(local_path, temporary)
        temporary.replace(remote_path)
        copied = 1
    else:
        copied = 0
    assert sha256_file(remote_path) == local_hash, f'Drive verification failed: {remote_path}'
    return copied

output_paths = (
    Path('features/prnu_v2_runtime_features.csv'),
    Path('features/prnu_v2_runtime_extraction_report.json'),
    Path('prnu_v2_runtime'),
    Path('rine_prnu_v2'),
    Path('reports/prnu_v2'),
)
copied = 0
verified = 0
for relative in output_paths:
    local_path = ROBUSTNESS / relative
    assert local_path.exists(), f'Expected Notebook 08 output missing: {local_path}'
    files = [local_path] if local_path.is_file() else sorted(
        path for path in local_path.rglob('*') if path.is_file()
    )
    for file_path in files:
        remote_path = DURABLE / file_path.relative_to(ROBUSTNESS)
        copied += copy_file_verified(file_path, remote_path)
        verified += 1
print(f'PASS: verified {verified} output files in {DURABLE}; copied {copied} new/changed files.')
print('The 19,460-image transform cache remains Colab-local by design.')
